# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [97]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [98]:
df = pd.read_csv('data/AviationData.csv', encoding='latin1', low_memory=False)

In [99]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  object 
 1   Investigation.Type      88889 non-null  object 
 2   Accident.Number         88889 non-null  object 
 3   Event.Date              88889 non-null  object 
 4   Location                88837 non-null  object 
 5   Country                 88663 non-null  object 
 6   Latitude                34382 non-null  object 
 7   Longitude               34373 non-null  object 
 8   Airport.Code            50132 non-null  object 
 9   Airport.Name            52704 non-null  object 
 10  Injury.Severity         87889 non-null  object 
 11  Aircraft.damage         85695 non-null  object 
 12  Aircraft.Category       32287 non-null  object 
 13  Registration.Number     87507 non-null  object 
 14  Make                    88826 non-null

In [100]:
df.describe()

,Number.of.Engines,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
count,82805.000000,77488.000000,76379.000000,76956.000000,82977.000000
mean,1.146585,0.647855,0.279881,0.357061,5.325440
std,0.446510,5.485960,1.544084,2.235625,27.913634
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,0.000000,1.000000
75%,1.000000,0.000000,0.000000,0.000000,2.000000
max,8.000000,349.000000,161.000000,380.000000,699.000000


In [101]:
df.columns

Index(['Event.Id', 'Investigation.Type', 'Accident.Number', 'Event.Date',
       'Location', 'Country', 'Latitude', 'Longitude', 'Airport.Code',
       'Airport.Name', 'Injury.Severity', 'Aircraft.damage',
       'Aircraft.Category', 'Registration.Number', 'Make', 'Model',
       'Amateur.Built', 'Number.of.Engines', 'Engine.Type', 'FAR.Description',
       'Schedule', 'Purpose.of.flight', 'Air.carrier', 'Total.Fatal.Injuries',
       'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured',
       'Weather.Condition', 'Broad.phase.of.flight', 'Report.Status',
       'Publication.Date'],
      dtype='object')

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [102]:
#Inspecting relevant columns
print(df["Aircraft.Category"].value_counts(dropna=False))
print('\n')
print(df["Investigation.Type"].value_counts(dropna=False))
print('\n')
print(df["Amateur.Built"].value_counts(dropna=False))

Aircraft.Category
NaN                  56602
Airplane             27617
Helicopter            3440
Glider                 508
Balloon                231
Gyrocraft              173
Weight-Shift           161
Powered Parachute       91
Ultralight              30
Unknown                 14
WSFT                     9
Powered-Lift             5
Blimp                    4
UNK                      2
Rocket                   1
ULTR                     1
Name: count, dtype: int64


Investigation.Type
Accident    85015
Incident     3874
Name: count, dtype: int64


Amateur.Built
No     80312
Yes     8475
NaN      102
Name: count, dtype: int64


In [103]:
# Filter Aircraft.Category to 'Airplane' only
# convert col to string, strip whitespace, and convert to lowercase
df['Aircraft.Category'] = df['Aircraft.Category'].astype(str).str.strip().str.lower()
df = df[df['Aircraft.Category'] == 'airplane']


# Filter Amateur.Built to 'No', professional builds only
# convert col to string, strip whitespace, and convert to lowercase
df['Amateur.Built'] = df['Amateur.Built'].astype(str).str.strip().str.lower()
df = df[df['Amateur.Built'] == 'no']                   

df = df[df["Investigation.Type"] == "Accident"]

# Convert Event.Date to datetime
df['Event.Date'] = pd.to_datetime(df['Event.Date'], errors='coerce')
# Filter Active/Recent aircraft (1983 onwards)
df = df[df['Event.Date'].dt.year >= 1983].copy()

df.head()

,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
4171,20001214X42331,Accident,ATL83FA140,1983-03-20,"CROSSVILLE, TN",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,1.0,1.0,NaN,NaN,IMC,Cruise,Probable Cause,02-05-2011
4285,20001214X42672,Accident,FTW83LA177,1983-04-02,"MCKINNEY, TX",United States,NaN,NaN,TX05,AERO COUNTRY,...,Skydiving,NaN,1.0,NaN,NaN,4.0,VMC,Standing,Probable Cause,17-10-2016
5960,20001214X44100,Accident,DCA83AA036,1983-08-21,"SILVANA, WA",United States,NaN,NaN,S88,NaN,...,Skydiving,NaN,11.0,2.0,NaN,13.0,VMC,Other,Probable Cause,17-10-2016
6669,20001214X44944,Accident,NYC84LA015,1983-10-28,"MIDDLETOWN, PA",United States,NaN,NaN,NaN,NaN,...,Unknown,NaN,1.0,NaN,NaN,29.0,VMC,Climb,Probable Cause,09-12-2011
6806,20001214X45188,Accident,NYC84LA028,1983-11-13,"MARTHA'S VINEYARD, MA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,NaN,NaN,NaN,1.0,VMC,Climb,Probable Cause,05-05-2011


## Data Cleaning

### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [104]:
# Impute missing injury columns with 0
injury_cols = ['Total.Fatal.Injuries', 'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured']
for col in injury_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
print(df[injury_cols].isnull().sum())

df['Total.Passengers'] = df[injury_cols].sum(axis=1)


# Calculate serious/fatal risk fraction (Avoid division by zero)
df['Fatal_Serious_Count'] = df['Total.Fatal.Injuries'] + df['Total.Serious.Injuries']


Total.Fatal.Injuries      0
Total.Serious.Injuries    0
Total.Minor.Injuries      0
Total.Uninjured           0
dtype: int64


**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [105]:
#Inspecting Aircraft.Damage column

df["Aircraft.damage"].value_counts(dropna=False)

Aircraft.damage
Substantial    16965
Destroyed       2316
NaN              434
Minor            147
Unknown           75
Name: count, dtype: int64

In [106]:

# Standardize aircraft damage categories (1 = Destroyed, 0 = Otherwise)
df['Aircraft.damage'] = df['Aircraft.damage'].fillna('Unknown').str.strip().str.title()

#Creating a destroyed/not destroyed metric
df["Aircraft_Destroyed"] = (
    df["Aircraft.damage"] == "Destroyed"
).astype(int)

df["Aircraft_Destroyed"].value_counts()

Aircraft_Destroyed
0    17621
1     2316
Name: count, dtype: int64

Cleaning assumptions: Missing values in the injury columns were replaced with 0 because they represent counts of people injured. The Aircraft.damage column was standardized, and unknown values were labeled "Unknown". Two derived variables were created:

1. Fatal_Serious_Injuries: the sum of fatal and serious injuries, providing a single measure of accident severity.
2. Aircraft_Destroyed: a binary indicator where 1 means the aircraft was destroyed and 0 means it was not destroyed.

### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [107]:
# Clean Make Column: strip whitespace, convert to uppercase to remove duplicate entries
df['Make'] = df['Make'].astype(str).str.strip().str.upper().fillna("Unknown")

# Filter: Keep only Makes with at least 50 historical accident entries
make_counts = df['Make'].value_counts()
valid_makes = make_counts[make_counts >= 50].index
df = df[df['Make'].isin(valid_makes)]

df["Make"].value_counts()

Make
CESSNA                            7029
PIPER                             3938
BEECH                             1392
BOEING                             553
MOONEY                             358
BELLANCA                           217
AIR TRACTOR INC                    217
MAULE                              215
CIRRUS DESIGN CORP                 209
AIR TRACTOR                        206
AERONCA                            200
CHAMPION                           157
GRUMMAN                            146
LUSCOMBE                           141
CIRRUS                             130
STINSON                            129
NORTH AMERICAN                     104
EMBRAER                             98
DEHAVILLAND                         93
TAYLORCRAFT                         93
AERO COMMANDER                      90
AIRBUS                              82
AVIAT AIRCRAFT INC                  76
SOCATA                              74
DIAMOND AIRCRAFT IND INC            72
AVIAT               

### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [108]:
#Check for missing models
df["Model"].isnull().sum()

# Clean Model Column: standardize format
df['Model'] = df['Model'].astype(str).str.strip().str.upper()

#Inspecting common models
df["Model"].value_counts().head(20)


Model
172          763
152          311
182          300
172S         272
PA28         268
172N         249
SR22         228
180          212
A36          181
172M         179
150          177
PA-18-150    175
PA-28-140    169
172P         141
737          118
140          117
172R         108
170B         107
PA-28-180    105
PA-28-161    101
Name: count, dtype: int64

In [109]:
#check whether model names are unique across manufactures

df.groupby("Model")["Make"].nunique().sort_values(ascending=False).head(20)


Model
NAN      6
7AC      3
8KCAB    3
8GCBC    3
7GCAA    3
7ECA     3
7GCBC    3
7EC      3
S2R      3
402A     2
C90A     2
A 1      2
500      2
200      2
G36      2
B36TC    2
HUSKY    2
A36      2
B300     2
400A     2
Name: Make, dtype: int64

In [110]:
# Create a unique plane type identifier combining standardized Make + Model strings
df['Plane_Type'] = (
    df['Make'].str.upper().str.strip() + 
    " " + 
    df['Model'].str.upper().str.strip()
)

df["Plane_Type"].value_counts().head(20)

Plane_Type
CESSNA 172                 763
CESSNA 152                 311
CESSNA 182                 300
CESSNA 172S                272
PIPER PA28                 268
CESSNA 172N                249
CESSNA 180                 212
CESSNA 172M                179
CESSNA 150                 177
PIPER PA-18-150            175
PIPER PA-28-140            169
BEECH A36                  164
CESSNA 172P                141
CIRRUS DESIGN CORP SR22    137
BOEING 737                 118
CESSNA 140                 116
CESSNA 172R                108
CESSNA 170B                107
PIPER PA-28-180            105
PIPER PA-28-161            101
Name: count, dtype: int64

#### Cleaning Tasks:
- **Handle Missing/Type Issues**: Convert the column to string type.
- **Remove Whitespace**: Use `.str.strip()` to eliminate leading and trailing spaces that create accidental duplicates.
- **Standardize Casing**: Convert all entries to uppercase (`.str.upper()`) so variations like "Boeing" and "BOEING" merge.
- **Filter Rare Makes**: Exclude low-frequency aircraft makes by keeping only those with at least 50 historical accident entries.
- **Standardize Models**: Clean and capitalize the `Model` column, then concatenate it with `Make` to create a unique `Plane_Type` identifier.


### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [111]:
print(
    df["Engine.Type"].unique(),
    df["Weather.Condition"].unique(),
    df["Purpose.of.flight"].unique(),
    df["Broad.phase.of.flight"].unique(),
    df["Number.of.Engines"].unique()
)

#Inspecting each column
columns_to_check = (
    "Engine.Type",
    "Weather.Condition",
    "Number.of.Engines",
    "Purpose.of.flight",
    "Broad.phase.of.flight"
)

for col in columns_to_check:
    
    print(df[col].value_counts(dropna=False))

['Reciprocating' 'Turbo Prop' 'Turbo Jet' nan 'Unknown' 'Turbo Fan'
 'Turbo Shaft' 'UNK'] ['IMC' 'VMC' 'UNK' nan 'Unk'] ['Personal' 'Skydiving' 'Unknown' 'Aerial Application' 'Positioning'
 'Instructional' nan 'Ferry' 'Other Work Use' 'Business' 'Public Aircraft'
 'Aerial Observation' 'Executive/corporate' 'Public Aircraft - Federal'
 'Air Race/show' 'Flight Test' 'Public Aircraft - State' 'Glider Tow'
 'Banner Tow' 'Air Race show' 'Firefighting' 'Public Aircraft - Local'
 'Air Drop' 'PUBS' 'ASHO'] ['Cruise' 'Standing' 'Climb' 'Takeoff' 'Landing' 'Approach' 'Maneuvering'
 'Descent' nan 'Taxi' 'Go-around' 'Unknown' 'Other'] [ 1.  2.  4. nan  0.  3.]
Engine.Type
Reciprocating    12736
NaN               2451
Turbo Prop         877
Turbo Fan          365
Unknown             67
Turbo Jet           45
Turbo Shaft          8
UNK                  1
Name: count, dtype: int64
Weather.Condition
VMC    13979
NaN     1509
IMC      872
Unk      147
UNK       43
Name: count, dtype: int64
Number.of.En

In [112]:
#Cleaning text colummns
#Removing whitespaces and standarding text

text_cols = [
    "Engine.Type",
    "Weather.Condition",
    "Purpose.of.flight",
    "Broad.phase.of.flight"
]
for col in text_cols:
    df[col]= (
        df[col].
            str.strip()
            .str.upper()
)

#Cleaning nummber of engines column

df["Number.of.Engines"] = pd.to_numeric(
    df["Number.of.Engines"], errors="coerce"
)

#Verifying
for col in columns_to_check:
    print(df[col].value_counts(dropna=False).head(20))

Engine.Type
RECIPROCATING    12736
NaN               2451
TURBO PROP         877
TURBO FAN          365
UNKNOWN             67
TURBO JET           45
TURBO SHAFT          8
UNK                  1
Name: count, dtype: int64
Weather.Condition
VMC    13979
NaN     1509
IMC      872
UNK      190
Name: count, dtype: int64
Number.of.Engines
1.0    13138
2.0     1940
NaN     1425
4.0       28
3.0       15
0.0        4
Name: count, dtype: int64
Purpose.of.flight
PERSONAL                     9776
INSTRUCTIONAL                2387
NaN                          1911
AERIAL APPLICATION            723
BUSINESS                      394
UNKNOWN                       255
POSITIONING                   254
SKYDIVING                     156
AERIAL OBSERVATION            146
OTHER WORK USE                118
BANNER TOW                     86
FERRY                          70
FLIGHT TEST                    68
EXECUTIVE/CORPORATE            59
GLIDER TOW                     29
PUBLIC AIRCRAFT - FEDERAL      2

### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [113]:
#Check for missing values
missing = df.isnull().sum().sort_values(ascending=False)

#Check missing values % of each column
missing_percent = (
    df.isnull().mean()*100
).sort_values(ascending=False)

pd.DataFrame({
    "Missing Values" : missing,
    "Percent Missing" : missing_percent.round(2)
})

,Missing Values,Percent Missing
Schedule,15176,91.70
Broad.phase.of.flight,14127,85.36
Air.carrier,8987,54.30
Airport.Code,5431,32.82
Airport.Name,5358,32.37
Report.Status,2934,17.73
Engine.Type,2451,14.81
Purpose.of.flight,1911,11.55
Weather.Condition,1509,9.12
Number.of.Engines,1425,8.61


In [114]:
#Droping columns
columns_to_drop = [
    "Schedule", "Air.carrier", "Airport.Code", "Airport.Name", "Latitude", "Longitude"
]
df = df.drop(columns=columns_to_drop, errors="ignore")

df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 16550 entries, 4171 to 88886
Data columns (total 29 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   Event.Id                16550 non-null  object        
 1   Investigation.Type      16550 non-null  object        
 2   Accident.Number         16550 non-null  object        
 3   Event.Date              16550 non-null  datetime64[ns]
 4   Location                16547 non-null  object        
 5   Country                 16549 non-null  object        
 6   Injury.Severity         16366 non-null  object        
 7   Aircraft.damage         16550 non-null  object        
 8   Aircraft.Category       16550 non-null  object        
 9   Registration.Number     16427 non-null  object        
 10  Make                    16550 non-null  object        
 11  Model                   16550 non-null  object        
 12  Amateur.Built           16550 non-null  object  

##### Column Filtering & Removal
Columns with substantial missing data and low strategic relevance were excluded. Specifically, Schedule, Air.carrier, Airport.Code, Airport.Name, Latitude, and Longitude were removed due to high missingness or limited utility in evaluating aircraft safety. Conversely, critical risk factors—including Weather.Condition, Engine.Type, Purpose.of.flight, Number.of.Engines, and Broad.phase.of.flight—were retained despite missing values, as they provide essential context for understanding accident outcomes.

### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [115]:
df.to_csv("data/AviationData_Cleaned.csv", index=False, encoding="latin1")
print("File saved successfully!")

File saved successfully!
